Let's do PCA analysis.

The first step is to make sure that there are no missing data in the dataset. To do this, we need to spatially interpolate the missing data using the calculated decorrelation scale.

Because each station has different decorrelation scales, we need to interpolate the missing data for each station separately.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os
import sys
from pathlib import Path

# Ensure local scripts are importable
project_root = Path("../../").resolve()
sys.path.append(str(project_root))

from scripts.data_loader import load_and_process_data, load_raw_shapefiles
from scripts.paths import ProjectPaths

print("Project Root:", project_root)

# Set force_reload=True if you added new raw data or changed the cleaning logic
merged_gdf = load_and_process_data(force_reload=True)

# Filter summer bottom do data
summer_bottom_df = merged_gdf[
    (merged_gdf['season'] == 'Summer') &
    (merged_gdf['saisuipoin'] == 2)
].dropna(subset=['do_']).copy()

# Convert do_ to numeric
summer_bottom_df['do_'] = pd.to_numeric(summer_bottom_df['do_'], errors='coerce')

Project Root: /home/reyha/SIS_Analysis_WSL
Loading raw shapefiles from 1981 to 2023...
Merging 86 shapefiles...
Processing data (cleaning DO, adding seasons)...
Saving processed data to /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...


In [3]:
# Parameters
weighting_func = 'gaussian'
min_weight = 0.01
search_radius_factor = 2.0
subset_dir = str(ProjectPaths.get_subset_dir('Summer', 'bottom'))  # default subset

In [ ]:
# Load datasets from 08_Decorrelation_Analysis_v3.ipynb
from pathlib import Path
subset_path = Path(subset_dir)

yearly_do_df = pd.read_csv(subset_path / "yearly_do_df.csv", index_col=0)
yearly_do_df.index = yearly_do_df.index.astype(str).str.zfill(8)

dist_matrix_km = pd.read_csv(ProjectPaths.PROCESSED_DATA_DIR / "dist_matrix_km.csv", index_col=0)
dist_matrix_km.index = dist_matrix_km.index.astype(str).str.zfill(8)
dist_matrix_km.columns = dist_matrix_km.columns.astype(str).str.zfill(8)

decorrelation_df = pd.read_csv(subset_path / "stations_decorrelation_scale.csv", index_col=0)
decorrelation_df.index = decorrelation_df.index.astype(str).str.zfill(8)

print(f"Loaded data from: {subset_path}")
print(f"Yearly DO: {yearly_do_df.shape}, Distance matrix: {dist_matrix_km.shape}, Decorrelation: {decorrelation_df.shape}")

Loaded data from: /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/summer_bottom
Yearly DO: (124, 41), Distance matrix: (124, 124), Decorrelation: (124, 7)


In [5]:
# Define weight functions

def exponential_weight(distance, scale):
    return np.exp(-distance / scale)

def gaussian_weight(distance, scale):
    return np.exp(-(distance**2) / (2 * scale**2))

In [6]:
# Define Interpolation function 

def interpolate_missing_do(yearly_do_df, decorrelation_df, dist_matrix, 
                            weighting_func='exponential', min_weight=0.01):
    """
    Interpolate missing DO values using neighboring stations.
    
    Search radius is set to 2x the station's decorrelation scale.
    Only neighbors within this radius are considered.
    """
    interpolated_df = yearly_do_df.copy()
    common_stations = list(set(yearly_do_df.index) & set(decorrelation_df.index) & set(dist_matrix.index))
    filled_count = 0
    
    for year in yearly_do_df.columns:
        for station in common_stations:
            if not pd.isna(yearly_do_df.loc[station, year]):
                continue
            scale = decorrelation_df.loc[station, 'Decorrelation_Scale_km']
            if pd.isna(scale) or scale <= 0:
                continue
            
            # Set search radius to 2x decorrelation scale
            search_radius = search_radius_factor * scale
            
            valid_stations = yearly_do_df.index[~pd.isna(yearly_do_df[year])].tolist()
            neighbors = [s for s in valid_stations if s in common_stations and s != station]
            if len(neighbors) == 0:
                continue
            
            distances = dist_matrix.loc[station, neighbors]
            
            # Filter neighbors within search radius
            within_radius = distances[distances <= search_radius]
            if len(within_radius) == 0:
                continue
            
            neighbors_in_radius = within_radius.index.tolist()
            
            # Get values and distances for neighbors in radius
            dist_vals = dist_matrix.loc[station, neighbors_in_radius].values  # numpy array
            do_vals = yearly_do_df.loc[neighbors_in_radius, year].values      # numpy array
            
            # Calculate weights (numpy operations)
            if weighting_func == 'exponential':
                weight_vals = exponential_weight(dist_vals, scale)
            else:
                weight_vals = gaussian_weight(dist_vals, scale)
            
            # Filter by minimum weight
            valid_mask = weight_vals > min_weight
            if valid_mask.sum() == 0:
                continue
            
            # Weighted average using numpy arrays (no index alignment issues)
            interpolated_df.loc[station, year] = (do_vals[valid_mask] * weight_vals[valid_mask]).sum() / weight_vals[valid_mask].sum()
            filled_count += 1
    
    print(f"Filled {filled_count} missing values (search radius = 2x decorrelation scale)")
    return interpolated_df

In [7]:
# Appy interpolation on data

missing_before = yearly_do_df.isna().sum().sum()
interpolated_do_df = interpolate_missing_do(yearly_do_df, decorrelation_df, dist_matrix_km, weighting_func='gaussian')
missing_after = interpolated_do_df.isna().sum().sum()

print(f"Before: {missing_before}, After: {missing_after}, Filled: {missing_before - missing_after}")

Filled 64 missing values (search radius = 2x decorrelation scale)
Before: 90, After: 26, Filled: 64


In [8]:
# Save interpolated data to the subset directory
from pathlib import Path
out_path = Path(subset_dir) / 'interpolated_do_df.csv'
interpolated_do_df.to_csv(out_path, index=True)
print(f"Saved interpolated DO to: {out_path}")

Saved interpolated DO to: /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/summer_bottom/interpolated_do_df.csv
